In [3]:
file_path = "../data/rawData/"
chinook_filename = f"{file_path}chinook.db"
northwind_filename = f"{file_path}northwind.db"

import os
os.environ["PREFECT_API_URL"] = "http://127.0.0.1:4200/api"

In [10]:
from sqlalchemy import create_engine
import pandas as pd

# Create an SQLAlchemy engine for the Chinook database
chinook_engine = create_engine(f"sqlite:///{chinook_filename}", echo=False)

# A connection can be explicitly opened and closed
chinook_conn = chinook_engine.connect()

# Or, as a best practice, use a 'with' statement for automatic resource management
with chinook_engine.connect() as conn:
    print("Connection to Chinook database established successfully.")
    # All database operations would happen here
    pass

chinook_conn.close() # Close the explicit connection

northwind_engine = create_engine(f"sqlite:///{northwind_filename}", echo=False)
northwind_conn = northwind_engine.connect()

with northwind_engine.connect() as conn:
    print("Connection to Nortwind database established successfully.")
    # All database operations would happen here
    pass

northwind_conn.close() # Close the explicit connection

Connection to Chinook database established successfully.
Connection to Nortwind database established successfully.


In [11]:
from sqlalchemy import inspect
import sys

def list_columns_in_database(engine, db_name):
    """
    Connects to a database and prints a list of all tables and their columns.

    Args:
        engine (sqlalchemy.engine.base.Engine): The SQLAlchemy engine for the database.
        db_name (str): The name of the database (for printing purposes).
    """
    print("-" * 30)
    print(f"Listing tables and columns in '{db_name}':")
    try:
        inspector = inspect(engine)
        table_names = inspector.get_table_names()

        if table_names:
            for table_name in table_names:
                print(f"\nTable: {table_name}")
                columns = inspector.get_columns(table_name)
                for column in columns:
                    print(f"  - {column['name']} ({column['type']})")
        else:
            print("No tables found in this database.")

    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    finally:
        print("-" * 30)

# List tables and columns for Chinook database
list_columns_in_database(chinook_engine, "Chinook")

# List tables and columns for Northwind database
list_columns_in_database(northwind_engine, "Northwind")

------------------------------
Listing tables and columns in 'Chinook':

Table: Album
  - AlbumId (INTEGER)
  - Title (NVARCHAR(160))
  - ArtistId (INTEGER)

Table: Artist
  - ArtistId (INTEGER)
  - Name (NVARCHAR(120))

Table: Customer
  - CustomerId (INTEGER)
  - FirstName (NVARCHAR(40))
  - LastName (NVARCHAR(20))
  - Company (NVARCHAR(80))
  - Address (NVARCHAR(70))
  - City (NVARCHAR(40))
  - State (NVARCHAR(40))
  - Country (NVARCHAR(40))
  - PostalCode (NVARCHAR(10))
  - Phone (NVARCHAR(24))
  - Fax (NVARCHAR(24))
  - Email (NVARCHAR(60))
  - SupportRepId (INTEGER)

Table: Employee
  - EmployeeId (INTEGER)
  - LastName (NVARCHAR(20))
  - FirstName (NVARCHAR(20))
  - Title (NVARCHAR(30))
  - ReportsTo (INTEGER)
  - BirthDate (DATETIME)
  - HireDate (DATETIME)
  - Address (NVARCHAR(70))
  - City (NVARCHAR(40))
  - State (NVARCHAR(40))
  - Country (NVARCHAR(40))
  - PostalCode (NVARCHAR(10))
  - Phone (NVARCHAR(24))
  - Fax (NVARCHAR(24))
  - Email (NVARCHAR(60))

Table: Genre
  - 

In [13]:
import sys
print(sys.executable)

import duckdb
print(duckdb.__version__)

/workspaces/OmniCorpPipeline/.venv/bin/python
1.5.5


In [16]:
from prefect import task, flow

@task(persist_result=False) # สั่งไม่ให้ใช้ระบบบันทึกผลลัพธ์อัตโนมัติ
def my_transform_task():
    return "data"


In [1]:
import os
os.environ["PREFECT_API_URL"] = "http://127.0.0.1:4200/api"

from prefect import flow, task
from pydantic import BaseModel

from prefect import flow, task

@task
def my_task():
    return "Done"

@flow
def my_flow():
    # แนะนำให้ใช้ .submit() แทนการเรียก my_task() ตรง ๆ ใน Notebook
    future = my_task.submit()
    return future.result()

my_flow()

08:21:30.820 | INFO    | Flow run 'brawny-crayfish' - Beginning flow run 'brawny-crayfish' for flow 'my-flow'

08:21:30.828 | INFO    | Flow run 'brawny-crayfish' - View at http://127.0.0.1:4200/runs/flow-run/69f9f432-cd6e-46da-b00c-3a6efd33a3cb

08:21:30.868 | INFO    | Task run 'my_task-dee' - Finished in state Completed()

08:21:31.865 | INFO    | Flow run 'brawny-crayfish' - Finished in state Completed()

'Done'

In [4]:
import duckdb
import pandas as pd
from prefect import flow, task


@task(name="Extract and Transform Customer Dimension")
def create_dim_customers_df(
    chinook_path, northwind_path
) -> pd.DataFrame:
    """ดึงข้อมูล Customer จาก Chinook และ Northwind มารวมเป็น dim_customers DataFrame"""
    conn = duckdb.connect()

    # เชื่อมต่อไฟล์ database ทั้งสองแห่ง (ปรับ path ตามจริง)
    conn.execute(f"ATTACH '{chinook_path}' AS chinook;")
    conn.execute(f"ATTACH '{northwind_path}' AS northwind;")

    query = """
    WITH chinook_customers AS (
        SELECT 
            'Chinook' AS source_system,
            CAST(CustomerId AS VARCHAR) AS source_customer_id,
            Company AS company_name,
            TRIM(CONCAT(FirstName, ' ', LastName)) AS contact_name,
            NULL AS contact_title,
            Address AS address,
            City AS city,
            State AS region,
            PostalCode AS postal_code,
            Country AS country,
            Phone AS phone,
            Fax AS fax,
            Email AS email,
            CAST(SupportRepId AS VARCHAR) AS support_rep_id
        FROM chinook.Customer
    ),
    northwind_customers AS (
        SELECT 
            'Northwind' AS source_system,
            CustomerID AS source_customer_id,
            CompanyName AS company_name,
            ContactName AS contact_name,
            ContactTitle AS contact_title,
            Address AS address,
            City AS city,
            Region AS region,
            PostalCode AS postal_code,
            Country AS country,
            Phone AS phone,
            Fax AS fax,
            NULL AS email,
            NULL AS support_rep_id
        FROM northwind.Customers
    ),
    unified_customers AS (
        SELECT * FROM chinook_customers
        UNION ALL
        SELECT * FROM northwind_customers
    )
    SELECT 
        ROW_NUMBER() OVER () AS customer_key, -- Surrogate Key สำหรับ Dimension Table
        *
    FROM unified_customers;
    """

    # ประมวลผลด้วย DuckDB แล้วแปลงเป็น Pandas DataFrame
    dim_customers = conn.execute(query).df()
    conn.close()

    return dim_customers


@flow(name="ETL Customer Dimension Flow")
def customer_dimension_pipeline():
    # เรียกใช้ Task สร้าง dim_customers DataFrame
    dim_customers_df = create_dim_customers_df(
        chinook_filename, northwind_filename
    )

    return dim_customers_df



df = customer_dimension_pipeline()
print(df.head())

08:30:39.634 | INFO    | Flow run 'prehistoric-scorpion' - Beginning flow run 'prehistoric-scorpion' for flow 'ETL Customer Dimension Flow'

08:30:39.636 | INFO    | Flow run 'prehistoric-scorpion' - View at http://127.0.0.1:4200/runs/flow-run/e9c94658-0f6d-4088-b10c-c6e4c82813b7

08:30:41.204 | INFO    | Task run 'Extract and Transform Customer Dimension-eb5' - Finished in state Completed()

08:30:41.657 | INFO    | Flow run 'prehistoric-scorpion' - Finished in state Completed()

   customer_key source_system source_customer_id  \
0             1       Chinook                  1   
1             2       Chinook                  2   
2             3       Chinook                  3   
3             4       Chinook                  4   
4             5       Chinook                  5   

                                       company_name           contact_name  \
0  Embraer - Empresa Brasileira de Aeronáutica S.A.         Luís Gonçalves   
1                                              None          Leonie Köhler   
2                                              None      François Tremblay   
3                                              None           Bjørn Hansen   
4                                  JetBrains s.r.o.  František Wichterlová   

  contact_title                          address                 city region  \
0          None  Av. Brigadeiro Faria Lima, 2170  São José dos Campos     SP   
1          None          Theodor-Heuss-Straße 34          